# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. `20 10`: `s["b"]` берёт значение по ярлыку `"b"`, а `s.iloc[0]` — по позиции, то есть первое значение.
2. `Алматы NaN`, `Астана NaN`, `Шымкент 3.0` (dtype `float64`): Pandas складывает **по ярлыкам**. Шымкент есть в обеих Series (2 + 1 = 3), Алматы и Астана — только в одной, поэтому `NaN`. Из-за `NaN` числа становятся `float`, а ярлыки идут по алфавиту.
3. `(2, 2)`: условию `x > 1` подходят две строки (x = 2 и x = 3), столбцов по-прежнему два.
4. `Series DataFrame`: одни скобки дают один столбец (Series), список столбцов в двойных скобках — таблицу (DataFrame).
5. `[1, 2] 5`: фильтр сохраняет старые ярлыки 1 и 2, а `iloc[0, 1]` берёт первую строку по позиции (x = 2) и второй столбец `y`, то есть 5.

In [1]:
# Проверяем ответы после объяснений выше
import pandas as pd

s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])

20 10
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
(2, 2)
Series DataFrame
[1, 2] 5


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [5]:
# 1
print("Строк и столбцов:", df.shape)
print(df.dtypes)
print("Пропусков в delivery_days:", df["delivery_days"].isna().sum())
for col in ["city", "category", "status", "customer_id"]:
    print(f"Разных значений в {col}: {df[col].nunique()}")

print("Уникальных order_id:", df["order_id"].nunique(), "из", len(df), "строк")
print("Заказов клиента TEST:", (df["customer_id"] == "TEST").sum())
print("Написания городов:", df["city"].unique())

Строк и столбцов: (5463, 8)
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object
Пропусков в delivery_days: 160
Разных значений в city: 14
Разных значений в category: 5
Разных значений в status: 2
Разных значений в customer_id: 2003
Уникальных order_id: 5283 из 5463 строк
Заказов клиента TEST: 3
Написания городов: <StringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'алматы',    'Атырау',
   'шымкент',  'Shymkent',    'астана',    'Актобе',   'Astana ',    'АЛМАТЫ',
   ' Алматы',    'Almaty']
Length: 14, dtype: str


**Вывод для менеджера.** В таблице 5 463 строки и 8 столбцов. `amount` был текстом (`"15 990"`), мы перевели его в числа готовой строкой. В `delivery_days` 160 пропусков. Городов 14 «разных», категорий 5, статусов 2, клиентов 2 003.

Три проблемы данных:
1. **Город записан 14 способами** (`Алматы`, `алматы`, `АЛМАТЫ`, `Almaty`, ` Алматы`…), хотя городов 6. Отчёт по одному написанию занизит выручку города: например, Алматы потеряет больше 250 заказов.
2. **Тестовый клиент TEST** — 3 заказа по 9 999 990 ₸. Он добавляет ~30 млн фальшивой выручки и ломает средний чек.
3. **Дубли заказов**: строк 5 463, а разных `order_id` только 5 283, то есть 180 повторов. Каждый дубль считает одну и ту же выручку дважды.

(Ещё: суммы, записанные текстом, без перевода в числа склеиваются в строку вместо сложения, а пропуски в доставке дают неверную долю «быстрых» доставок, если делить на все строки.)

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [6]:
# 2
amount_clean = df.loc[paid & not_test, "amount"]
print("Заказов:", len(amount_clean))
print(f"Среднее:  {amount_clean.mean():,.0f} ₸")
print(f"Медиана:  {amount_clean.median():,.0f} ₸")
print(f"Q1 (25%): {amount_clean.quantile(0.25):,.0f} ₸")
print(f"Q3 (75%): {amount_clean.quantile(0.75):,.0f} ₸")
print(f"Менеджеру: типичный заказ стоит около {amount_clean.median():,.0f} ₸; "
      f"половина заказов лежит между {amount_clean.quantile(0.25):,.0f} и {amount_clean.quantile(0.75):,.0f} ₸.")

Заказов: 4901
Среднее:  14,672 ₸
Медиана:  10,300 ₸
Q1 (25%): 5,880 ₸
Q3 (75%): 18,690 ₸
Менеджеру: типичный заказ стоит около 10,300 ₸; половина заказов лежит между 5,880 и 18,690 ₸.


**Вывод для менеджера.** Типичный чек — **медиана ≈ 10 300 ₸**, а половина всех заказов стоит от 5 880 до 18 690 ₸. Среднее (≈ 14 672 ₸) заметно выше, потому что его тянут вверх редкие дорогие заказы электроники. В отчёте честнее показывать **медиану** (вместе с Q1–Q3): она описывает обычного покупателя и почти не меняется от нескольких крупных заказов. Среднее нужно для расчёта выручки (среднее × число заказов), но не для описания «типичного» клиента.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [7]:
# 3
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
cancelled = df["status"] == "отменён"
is_sh = df["city"].isin(SH)
is_al = df["city"].isin(AL)

# Среднее от True/False = доля True
share_all = cancelled.mean()
share_sh = cancelled[is_sh].mean()
share_al = cancelled[is_al].mean()

print(f"Все заказы: {share_all:.1%} ({len(df)} заказов)")
print(f"Шымкент:    {share_sh:.1%} ({is_sh.sum()} заказов)")
print(f"Алматы:     {share_al:.1%} ({is_al.sum()} заказов)")
print(f"Менеджеру: в Шымкенте отменяют {share_sh / share_al:.1f} раза чаще, чем в Алматы, "
      f"и {share_sh / share_all:.1f} раза чаще, чем в среднем.")

Все заказы: 10.2% (5463 заказов)
Шымкент:    20.4% (804 заказов)
Алматы:     7.9% (2188 заказов)
Менеджеру: в Шымкенте отменяют 2.6 раза чаще, чем в Алматы, и 2.0 раза чаще, чем в среднем.


**Вывод для менеджера.** Выделяется **Шымкент**: там отменён каждый пятый заказ (≈ 20,4%), а в Алматы — около 7,9%, то есть в **≈ 2,6 раза** реже. По всем городам доля отмен ≈ 10,2%, Шымкент выше среднего в два раза. Стоит разобраться с причинами отмен в Шымкенте (доставка, наличие товара, работа курьеров).

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [8]:
# 4
base = paid & not_test
changes = {}
for name, spellings in [("Шымкент", SH), ("Алматы", AL)]:
    city = base & df["city"].isin(spellings)
    rev_sep = df.loc[city & sep, "amount"].sum()
    rev_oct = df.loc[city & octo, "amount"].sum()
    changes[name] = (rev_oct - rev_sep) / rev_sep * 100
    print(f"{name}: сентябрь {rev_sep:,.0f} ₸, октябрь {rev_oct:,.0f} ₸, изменение {changes[name]:.1f}%")

print(f"Менеджеру: в Шымкенте падение в {changes['Шымкент'] / changes['Алматы']:.1f} раза сильнее, чем в Алматы.")

Шымкент: сентябрь 4,949,800 ₸, октябрь 4,044,010 ₸, изменение -18.3%
Алматы: сентябрь 15,523,940 ₸, октябрь 14,643,020 ₸, изменение -5.7%
Менеджеру: в Шымкенте падение в 3.2 раза сильнее, чем в Алматы.


**Вывод для менеджера.** Выручка Шымкента упала примерно на **18,3%** (4,95 → 4,04 млн ₸), Алматы — на **5,7%** (15,52 → 14,64 млн ₸). Падение в Шымкенте сильнее в **≈ 3,2 раза**. Значит, проблема не общая для всего магазина, а в основном сосредоточена в Шымкенте — вместе с высокой долей отмен из задания 3. Цифры ещё с дублями, поэтому точные проценты уточним после очистки на уроке 6, но направление вывода не изменится.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [9]:
# 5
# Ошибка 1 (роняет код): нет скобок вокруг условий. `&` выполняется раньше `==`,
# поэтому Python пытается вычислить "Шымкент" & df["status"] и падает с TypeError.
try:
    cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
except TypeError as e:
    print("Ошибка 1:", type(e).__name__)

# Ошибка 2 (тихо неверно): == "Шымкент" берёт только одно написание из трёх.
only_one = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
print("Только 'Шымкент':", len(only_one))

# Исправленный код: скобки + isin со всеми написаниями
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print("Все написания:", len(cancel_sh))

Ошибка 1: TypeError
Только 'Шымкент': 153
Все написания: 164


**Вывод.** Первая ошибка — нет скобок: `&` выполняется раньше `==`, код падает с `TypeError`. Вторая — сравнение только с `"Шымкент"`: код работает, но теряет «шымкент» и «Shymkent» и показывает **153** отмены вместо **164**. Исправление: каждое условие в скобках и `isin(SH)` для города.

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [10]:
# 6
top_clients = df["customer_id"].value_counts()
print(top_clients.head())
top = top_clients.index[0]
print("Самый частый клиент:", top)

mask = df["customer_id"] == top
df.loc[mask, ["order_id", "order_date", "city", "amount", "status"]]

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64
Самый частый клиент: C01818


,order_id,order_date,city,amount,status
57,101125,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,100096,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,101860,2026-09-24 18:00,Астана,1490.0,оплачен
2617,101462,2026-09-29 09:00,Астана,22170.0,оплачен
2776,101495,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,103416,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,102644,2026-10-18 08:00,Астана,13600.0,оплачен
4610,104001,2026-10-21 19:00,Алматы,21990.0,оплачен


In [11]:
top_orders = df.loc[mask, "order_id"]
print("Строк у клиента:", len(top_orders), "| разных order_id:", top_orders.nunique())
# duplicated(keep=False) отмечает True все копии повторяющегося номера, чтобы увидеть пару целиком
print(df.loc[mask & df["order_id"].duplicated(keep=False), ["order_id", "order_date", "amount", "status"]])
print("Во всей таблице повторов order_id:", len(df) - df["order_id"].nunique())

Строк у клиента: 11 | разных order_id: 10
      order_id        order_date  amount   status
1371    101504  2026-09-15 21:00  8890.0  оплачен
1375    101504  2026-09-15 21:00  8890.0  оплачен
Во всей таблице повторов order_id: 180


**Вывод для менеджера.** Больше всего строк у клиента **C01818** — 11. Но разных `order_id` у него только 10: один заказ записан дважды (та же дата, сумма и статус). Значит, на самом деле он сделал 10 заказов. По всей таблице таких повторов **180**. Вся выручка, которую мы считали на уроке (включая −8,2% после удаления TEST), **завышена** на сумму этих дублей. Перед отчётом дубли нужно удалить — это тема урока 6 (там получается −2,3%).

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [12]:
# 7
q1 = amount_clean.quantile(0.25)
q3 = amount_clean.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f"Q1 = {q1:,.0f}, Q3 = {q3:,.0f}, IQR = {iqr:,.0f}, граница = {upper:,.0f} ₸")

outliers = df[paid & not_test & (df["amount"] > upper)]
print(f"Выше границы: {len(outliers)} заказов, это {len(outliers) / len(amount_clean):.1%} от {len(amount_clean)}")
print(outliers["category"].value_counts())

Q1 = 5,880, Q3 = 18,690, IQR = 12,810, граница = 37,905 ₸
Выше границы: 320 заказов, это 6.5% от 4901
category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64


**Вывод для менеджера.** Граница IQR ≈ **37 905 ₸**. Выше неё **320 заказов** (≈ 6,5% оплаченных без TEST), и почти все они (308) — **Электроника**. Это не ошибки, а нормальные дорогие покупки техники.

Почему не 87, как по z-оценке в ДЗ 4: z-оценка считает границу как «среднее + 3 стандартных отклонения». Дорогие заказы сами раздувают среднее и стандартное отклонение, граница уходит высоко (≈ 54,6 тыс. ₸ против 37,9 тыс. у IQR), и выбросами становятся только самые крайние заказы. IQR строится по квартилям, на которые крупные заказы почти не влияют, и коэффициент 1,5 мягче, чем 3. Поэтому IQR отмечает больше заказов. Ни одно правило не «истинное»: выброс — это повод проверить заказ, а не удалить его.

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [13]:
# 8
cancel_counts = df.loc[cancelled, "city"].value_counts()
all_counts = df["city"].value_counts()
cancel_share = (cancel_counts / all_counts).sort_values(ascending=False)
print(cancel_share.round(3))

city
Shymkent     0.231
Шымкент      0.215
алматы       0.141
астана       0.131
Актобе       0.103
Astana       0.100
Караганда    0.089
Астана       0.085
Алматы       0.080
Атырау       0.075
Almaty       0.058
 Алматы      0.045
шымкент      0.037
АЛМАТЫ       0.032
Name: count, dtype: float64


In [14]:
# Проверка деления по ярлыкам: порядок строк в двух Series разный, но Pandas совмещает ярлыки
print("Порядок одинаковый?", list(cancel_counts.index) == list(all_counts.index))
print("Шымкент вручную:", cancel_counts["Шымкент"], "/", all_counts["Шымкент"], "=",
      round(cancel_counts["Шымкент"] / all_counts["Шымкент"], 3), "| из деления Series:", round(cancel_share["Шымкент"], 3))

check = pd.DataFrame({"отмен": cancel_counts[SH], "всего": all_counts[SH], "доля": cancel_share[SH].round(3)})
print(check)

Порядок одинаковый? False
Шымкент вручную: 153 / 711 = 0.215 | из деления Series: 0.215
          отмен  всего   доля
city                         
Шымкент     153    711  0.215
шымкент       2     54  0.037
Shymkent      9     39  0.231


**Вывод для менеджера.** Деление Series идёт по ярлыкам: порядок городов в двух `value_counts` разный, но доля для «Шымкент» совпадает с ручным расчётом (153 / 711 ≈ 0,215).

У написаний Шымкента доли очень разные: «Шымкент» ≈ 21,5% (711 заказов), «Shymkent» ≈ 23,1% (39 заказов), «шымкент» ≈ 3,7% (всего 54 заказа, 2 отмены). На маленьких группах доля прыгает от одной-двух отмен, поэтому **судить о городе по одному написанию нельзя** — это один город, просто данные записаны по-разному. До анализа нужно привести названия городов к одному виду: убрать пробелы по краям, привести регистр, заменить латиницу на кириллицу (`Almaty` → `Алматы`). Это тема урока 6.